In [1]:
# Extract full trajectories
import pandas as pd
import os

# 1. Define your targets (Experiment/Site, Track_ID)
# Example: [("Exp08_Site10", 580), ("Exp09_Site11", 948)]
targets = [
    # apo
    ("Exp09_Site09", 1321), 
    ("Exp09_Site05", 216),
    ("Exp08_Site05", 673),
    # ("Exp09_Site09", 1175),
    # ("Exp09_Site10", 798),
    # nuclear shrinkage
    #("Exp06_Site33", 1055),
    #("Exp06_Site45", 438),
    #("Exp06_Site33", 530),
    #("Exp06_Site33", 952),
    #("Exp06_Site21", 29),
    # mito
    #("Exp12_Site05", 1034), 
    ("Exp13_Site02", 1247),
    ("Exp09_Site10", 1187),
    ("Exp09_Site08", 1276),
    #("Exp10_Site05", 2673),

]

# 2. Define where your source data lives
# Adjust these paths to your actual system
FOV_DIR = "/mnt/imaging.data/PertzLab/apoDetection/TIFFs" 
TRACKING_DIR = "/home/nbahou/myimaging/apoDet_refactored/data/paolo_data_d5_c12/track_dfs"
OUTPUT_TRAJ_DIR = "./d5_c12_example_trajectories/"

# 3. Helper to load the specific tracking DF for a target
def get_track_df(filename):
    """
    Assumes tracking files are named something like 'Exp08_Site10_tracking.csv'
    Adjust the string formatting below to match your actual filenames.
    """
    path = os.path.join(TRACKING_DIR, f"{filename}.csv")
    if os.path.exists(path):
        return pd.read_csv(path)
    else:
        print(f"Warning: Tracking file not found at {path}")
        return None

print(f"Plan initialized for {len(targets)} tracks.")

Plan initialized for 6 tracks.


In [2]:
# --- Track Length Checker ---
stats = []

for filename, tid in targets:
    df_track = get_track_df(filename)
    if df_track is not None:
        # Filter for the specific ID
        track_subset = df_track[df_track['track_id'] == tid]
        
        if len(track_subset) == 0:
            stats.append({'filename': filename, 'track_id': tid, 'status': 'NOT FOUND'})
            continue
            
        t_min = track_subset['t'].min()
        t_max = track_subset['t'].max()
        n_frames = len(track_subset)
        duration = t_max - t_min
        
        # A valid window needs 55 units of time (11 gaps * 5 units)
        is_viable = duration >= 55
        
        stats.append({
            'filename': filename,
            'track_id': tid,
            'frames': n_frames,
            't_start': t_min,
            't_end': t_max,
            'total_span': duration,
            'viable': 'YES' if is_viable else 'TOO SHORT'
        })

# Display as a clean DataFrame
stats_df = pd.DataFrame(stats)
print("\n--- Trajectory Length Summary ---")
print(stats_df.to_string(index=False))

# Quick summary
viable_count = (stats_df['viable'] == 'YES').sum()
print(f"\nSummary: {viable_count}/{len(targets)} tracks are long enough for the 12-channel/5-step window.")


--- Trajectory Length Summary ---
    filename  track_id  frames  t_start  t_end  total_span viable
Exp09_Site09      1321     541        0    551         551    YES
Exp09_Site05       216     586        0    587         587    YES
Exp08_Site05       673    1450        0   1449        1449    YES
Exp13_Site02      1247    1238        0   1237        1237    YES
Exp09_Site10      1187     660        0    659         659    YES
Exp09_Site08      1276    1013        0   1014        1014    YES

Summary: 6/6 tracks are long enough for the 12-channel/5-step window.


In [3]:
import numpy as np
import tifffile as tiff
import os
import pandas as pd
from tqdm import tqdm

def run_trajectory_extraction(targets, fov_dir, tracking_dir, output_dir, crop_size=32, step=5):
    os.makedirs(output_dir, exist_ok=True)
    records = []
    
    for filename, tid in targets:
        print(f"\n--- Processing {filename} | Track {tid} ---")
        
        # 1. Load Data
        fov_path = os.path.join(fov_dir, f"{filename}.tif")
        track_path = os.path.join(tracking_dir, f"{filename}.csv")
        
        if not os.path.exists(fov_path) or not os.path.exists(track_path):
            print(f"Skipping: Files missing for {filename}")
            continue
            
        df_track = pd.read_csv(track_path)
        this_track = df_track[df_track['track_id'] == tid].sort_values('t')
        coords = this_track.set_index('t')[['x', 'y']].to_dict('index')
        
        with tiff.TiffFile(fov_path) as tif:
            stack = tif.asarray() # (T, H, W)
            T_max, H_max, W_max = stack.shape

        # 2. Rolling Window Extraction
        count = 0
        half = crop_size // 2
        all_ts = sorted(coords.keys())

        for t_start in tqdm(all_ts, desc=f"Extracting {filename}_{tid}"):
            # The 12 timepoints for the 12 channels
            target_ts = [t_start + (k * step) for k in range(12)]
            
            # Validity Checks
            if not all(t in coords for t in target_ts): continue
            if not all(0 <= t < T_max for t in target_ts): continue

            # Edge Check (using the center at t_start)
            y_c, x_c = coords[t_start]['y'], coords[t_start]['x']
            y1, y2, x1, x2 = int(y_c - half), int(y_c + half), int(x_c - half), int(x_c + half)
            
            if y1 < 0 or y2 > H_max or x1 < 0 or x2 > W_max:
                continue 

            # Create 12-channel crop
            crop = np.zeros((12, crop_size, crop_size), dtype=stack.dtype)
            for idx, t_curr in enumerate(target_ts):
                # Get the SPECIFIC centroid for this specific channel's timepoint
                curr_y_c, curr_x_c = coords[t_curr]['y'], coords[t_curr]['x']
                
                # Calculate bounds for THIS channel
                cy1, cy2 = int(curr_y_c - half), int(curr_y_c + half)
                cx1, cx2 = int(curr_x_c - half), int(curr_x_c + half)
                
                # Optional: Add a check to ensure these bounds are within stack limits
                if cy1 < 0 or cy2 > H_max or cx1 < 0 or cx2 > W_max:
                    # Handle edge cases (padding or skipping)
                    continue
            
                crop[idx] = stack[t_curr, cy1:cy2, cx1:cx2]

            # Save File
            out_name = f"traj_{filename}_ID{tid}_T{t_start:04d}.tif"
            out_path = os.path.join(output_dir, out_name)
            tiff.imwrite(out_path, crop)
            
            records.append({
                'file_path': os.path.abspath(out_path),
                'filename': filename,
                'track_id': tid,
                't_start': t_start,
                'y_center': y_c,
                'x_center': x_c,
                'window_size': crop_size,
                'class': 'trajectory' # Useful for filtering later
            })
            count += 1
            
        print(f"Done. Extracted {count} windows.")

    # Save a specific metadata file for these trajectories
    meta_df = pd.DataFrame(records)
    meta_path = os.path.join(output_dir, "metadata_dense_trajectories.csv")
    meta_df.to_csv(meta_path, index=False)
    return meta_path

# Execute
metadata_traj_path = run_trajectory_extraction(targets, FOV_DIR, TRACKING_DIR, OUTPUT_TRAJ_DIR)


--- Processing Exp09_Site09 | Track 1321 ---


Extracting Exp09_Site09_1321: 100%|██████████████████████████████████████████████████| 541/541 [00:04<00:00, 130.32it/s]


Done. Extracted 437 windows.

--- Processing Exp09_Site05 | Track 216 ---


Extracting Exp09_Site05_216: 100%|████████████████████████████████████████████████████| 586/586 [00:06<00:00, 92.96it/s]


Done. Extracted 519 windows.

--- Processing Exp08_Site05 | Track 673 ---


...

Done. Extracted 1183 windows.

--- Processing Exp09_Site10 | Track 1187 ---


Extracting Exp09_Site10_1187: 100%|██████████████████████████████████████████████████| 660/660 [00:04<00:00, 137.18it/s]


Done. Extracted 605 windows.

--- Processing Exp09_Site08 | Track 1276 ---


Extracting Exp09_Site08_1276: 100%|████████████████████████████████████████████████| 1013/1013 [00:08<00:00, 113.99it/s]


Done. Extracted 955 windows.
